In [2]:
#Step 1 : Load the data
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Open_ai.pdf")
data = loader.load()

In [3]:
data[0]

Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'author': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Open_ai.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and\ndocument classiﬁcation. Although large unlabeled text corpora are abundant,\nlabeled data for learning these speciﬁc tasks is scarce, making it challe

In [5]:
#Step 2 : Break the Data into chunks
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

chunks = RecursiveCharacterTextSplitter(chunk_size=1000)

#Split the data into documents
text_split = chunks.split_documents(data)

#print the len of text_split
print(f"Length of text split : {len(text_split)}")

Length of text split : 56


In [6]:
text_split[0]

Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'author': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Open_ai.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and\ndocument classiﬁcation. Although large unlabeled text corpora are abundant,\nlabeled data for learning these speciﬁc tasks is scarce, making it challe

In [13]:
#Step 3 : Convert the data to the embedding
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
#Or you can use
#from langchain_google_genai import GoogleGenerativeEmbeddings


## Create an embeddings using HF
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectors = embeddings.embed_query("Hello World!")

vectors[:5]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2873.88it/s]


[-0.02038683369755745,
 0.025280876085162163,
 -0.0005662108305841684,
 0.011615470983088017,
 -0.037988435477018356]

In [16]:
#Step 4 : Create Vectorstores to store the vectors
vector_stores = Chroma.from_documents(documents=data, embedding=embeddings)

In [18]:
#Step 5 : Create Reteriver  and invoke checke length
reteriver = vector_stores.as_retriever(search_type='similarity', search_kwargs={"k":10})
reterived_docs = reteriver.invoke("What are the two main stages of the training procedure proposed in the paper?")

print(f"Length of reterivd docs : {len(reterived_docs)}")

Length of reterivd docs : 10


In [23]:
reterived_docs[5].page_content

'Figure 1: (left) Transformer architecture and training objectives used in this work. (right) Input\ntransformations for ﬁne-tuning on different tasks. We convert all structured inputs into token\nsequences to be processed by our pre-trained model, followed by a linear+softmax layer.\n3.3 Task-speciﬁc input transformations\nFor some tasks, like text classiﬁcation, we can directly ﬁne-tune our model as described above.\nCertain other tasks, like question answering or textual entailment, have structured inputs such as\nordered sentence pairs, or triplets of document, question, and answers. Since our pre-trained model\nwas trained on contiguous sequences of text, we require some modiﬁcations to apply it to these tasks.\nPrevious work proposed learning task speciﬁc architectures on top of transferred representations [44].\nSuch an approach re-introduces a signiﬁcant amount of task-speciﬁc customization and does not\nuse transfer learning for these additional architectural components. Inste

In [24]:
#Step 6 : Create an LLM using the Generative Gemini model
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash",
    api_key = os.environ["GEMINI_API_KEY"],
    temperature = 0.3,
    max_tokens = 200
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [28]:
#Step 7 : prompt Template and chaining
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
#Step 8 : Create question_answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)

#Now we can make RAG chain
rag_chain = create_retrieval_chain(reteriver, question_answer_chain)

In [34]:
#Step 9 : print response by invoking the llm
response = rag_chain.invoke({"input" : "What dataset was used for the unsupervised pre-training phase?"})
print(response["answer"])

c:\Users\ADITHYA UBALE\miniconda3\envs\rag\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The context states that the unsupervised pre-
